In [ ]:
!ffmpeg -version

In [ ]:
!yt-dlp -x --audio-format wav --audio-quality 0 "https://www.youtube.com/watch?v=7Zws-tsnv5w" -o "gashyqtar1.wav"

In [ ]:
!ffmpeg -ss 00:00:00 -t 00:10:00 -i gashyqtar1.wav gashyqtar_10min.wav

In [ ]:
!demucs --two-stems=vocals gashyqtar_10min.wav

In [ ]:
# ============================================================
# CELL 1: Separate full audio + measure time
# ============================================================
import time

start = time.time()
!demucs --two-stems=vocals gashyqtar1.wav
elapsed = time.time() - start

print(f"\n{'='*40}")
print(f"Demucs processing time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"{'='*40}")

In [ ]:
# ============================================================
# CELL 2: Cut audio starting from 6 seconds for WER testing
# ============================================================

# Raw audio (with music) - skip first 6 seconds
!ffmpeg -ss 00:00:06 -i gashyqtar1.wav gashyqtar_raw_for_wer.wav -y

# Separated vocals - skip first 6 seconds
# (adjust path to match your actual demucs output folder name)
!ffmpeg -ss 00:00:06 -i "separated/htdemucs/gashyqtar1/vocals.wav" gashyqtar_sep_for_wer.wav -y

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="float16")

for label, path in [
    ("raw", "gashyqtar_raw_for_wer.wav"),
    ("sep", "gashyqtar_sep_for_wer.wav"),
]:
    segments, _ = model.transcribe(
        path,
        language="kk",
        condition_on_previous_text=False,
        compression_ratio_threshold=2.4,
        no_repeat_ngram_size=5,
    )
    text = " ".join([s.text for s in segments])
    with open(f"transcript_{label}.txt", "w", encoding="utf-8") as f:
        f.write(text)
    print(f"{label.upper()} transcription done → transcript_{label}.txt")

In [ ]:
!pip install jiwer -q

In [ ]:
# ============================================================
# CELL 4: Compute WER
# ============================================================

from jiwer import wer
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r'[«»„""\'\-—–]', '', text)   # remove quotes and dashes
    text = re.sub(r'[^\w\s]', '', text)           # remove other punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load reference (ground truth from book)
reference_raw = open("book_transcript_gashyktar001.txt", encoding="utf-8").read()

# Load ASR outputs
hyp_raw = open("transcript_raw.txt", encoding="utf-8").read()
hyp_sep = open("transcript_sep.txt", encoding="utf-8").read()

# Normalize all three
ref  = normalize(reference_raw)
h_raw = normalize(hyp_raw)
h_sep = normalize(hyp_sep)

wer_raw = wer(ref, h_raw)
wer_sep = wer(ref, h_sep)

print(f"{'='*40}")
print(f"WER with background music : {wer_raw:.3f} ({wer_raw*100:.1f}%)")
print(f"WER after separation      : {wer_sep:.3f} ({wer_sep*100:.1f}%)")
print(f"Relative improvement      : {(wer_raw - wer_sep) / wer_raw * 100:.1f}%")
print(f"{'='*40}")